# HealthGraph: ETL

Matrizes curriculares 2026 de Biomedicina, Enfermagem, Farmácia, Medicina, Medicina Veterinária e Odontologia (PDF) → pandas → Neo4j. Só disciplinas obrigatórias.

| Etapa | O que faz |
|---|---|
| 1. Extract | lê os PDFs de `assets/matrizes/` |
| 2. Transform | limpa os dados e monta uma tabela por nó e por relacionamento |
| 3. Load | cria as constraints e carrega com `MERGE` |
| 4. Validação | confere o grafo contra os PDFs |

## 0. Modelagem

```
(:Curso {codigo, nome, chObrigatoria, periodos})
  -[:OFERECE {periodo, tipo, chMinima, periodosCompletos}]->
(:Disciplina {codigo, nome, nomeBase, cht, chp, ches, chTotal, chex, desativada})
  -[:REQUER {tipo: 'pre' | 'co'}]-> (:Disciplina)
  -[:EQUIVALE_A]-> (:Disciplina)
  -[:DO_DEPTO]-> (:Departamento {sigla})
```

- `periodo` fica em `OFERECE` porque a mesma disciplina pode estar em períodos diferentes em cada curso.
- `Departamento` é nó para agrupar e navegar por ele.
- `EQUIVALE_A` vem de `data/equivalencias.csv`, revisado à mão, porque cada curso usa códigos próprios.

## 1. Extract

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from neo4j import GraphDatabase

NEO4J_URI  = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "neo4j12345"))
NEO4J_DB   = os.getenv("NEO4J_DATABASE", "neo4j")

PDF_DIR    = Path("assets/matrizes")
CYPHER_DIR = Path("cypher")
DATA_DIR   = Path("data")
OUT_DIR    = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
driver.verify_connectivity()

In [2]:
COLUNAS = ["codigo", "nome", "tipo", "cht", "chp", "ches", "chtotal", "chex", "chprereq", "prereq", "coreq"]
RE_PERIODO = re.compile(r"(\d+)º período")
RE_CODIGO  = re.compile(r"[A-Z]{3}\d{5}")

In [3]:
# codigo,nome e carga horaria obrigatoria do curso (1° pag)

def ler_cabecalho(pdf):
    txt = pdf.pages[0].extract_text()
    return{
        "codigo": re.search(r"Currículo:\s*([\d.]+)", txt).group(1),
        "nome": re.search(r"Curso:\s*(.+)", txt).group(1).strip(),
        "chObrigatoria": int(re.search(r"Carga horária obrigatória:\s*(\d+)", txt).group(1)),
    }

In [4]:
def config_tabela(pagina):
    verticais = pagina.vertical_edges
    
    if not verticais:
        return None
    
    borda = min(traco["x0"] for traco in verticais)
    alturas = []
    
    for traco in verticais:
        if traco["x0"] == borda:
            alturas += [traco["top"], traco["bottom"]]
    
    return {"vertical_strategy": "lines", "horizontal_strategy": "explicit", "explicit_horizontal_lines": alturas}

In [ ]:
# cada tabela pertence a parte "Nº período" que aparece acima dela

def periodo_tabela(pagina, tabela, periodo):
    topo_tabela = tabela.bbox[1]

    for linha in pagina.extract_text_lines():
        achou = RE_PERIODO.match(linha["text"])

        if achou and linha["top"] < topo_tabela:
            periodo = int(achou.group(1))

    return periodo

In [ ]:
def linhas_pdf(pdf):
    linhas = []
    periodo = None

    for pagina in pdf.pages:
        config = config_tabela(pagina)
        if config is None:
            continue

        for tabela in pagina.find_tables(config):
            periodo = periodo_tabela(pagina, tabela, periodo)

            for celulas in tabela.extract():
                textos = [(celula or "").replace("\n", " ") for celula in celulas]
                linha = dict(zip(COLUNAS, textos))
                linha["periodo"] = periodo
                linhas.append(linha)

    return linhas

In [7]:
def ler_disciplinas(pdf):
    disciplinas = []

    for linha in linhas_pdf(pdf):
        if RE_CODIGO.fullmatch(linha["codigo"]):
            disciplinas.append(linha)

        elif linha["codigo"] == "" and disciplinas:
            anterior = disciplinas[-1]
            for coluna in ["nome", "prereq", "coreq"]:
                anterior[coluna] = (anterior[coluna] + " " + linha[coluna]).strip()

    return disciplinas

In [8]:
# le todos os pdfs

cursos, linhas = [], []

for arquivo in sorted(PDF_DIR.glob("*.pdf")):
    with pdfplumber.open(arquivo) as pdf:
        curso = ler_cabecalho(pdf)
        disciplinas = ler_disciplinas(pdf)

        for disciplina in disciplinas:
            disciplina["curso"] = curso["codigo"]
            linhas.append(disciplina)

    cursos.append(curso)
    print(f"{curso['nome']}: {len(disciplinas)} disciplinas lidas")

df_bruto = pd.DataFrame(linhas)
print("Linhas extraídas:", df_bruto.shape)
df_bruto.head()

BIOMEDICINA: 176 disciplinas lidas
ENFERMAGEM: 106 disciplinas lidas
FARMÁCIA: 148 disciplinas lidas
MEDICINA: 240 disciplinas lidas
MEDICINA VETERINÁRIA: 183 disciplinas lidas
ODONTOLOGIA: 110 disciplinas lidas
Linhas extraídas: (963, 13)


,codigo,nome,tipo,cht,chp,ches,chtotal,chex,chprereq,prereq,coreq,periodo,curso
0,GAN00144,COMPLEMENTOS DE MATEMÁTICA APLICADA,OB,60,0,0,60,0,,,,1,48.01.002
1,GCM00016,BIOLOGIA CELULAR E MOLECULAR,OB,60,20,0,80,0,,,,1,48.01.002
2,GFI00110,FUNDAMENTOS DE FISICA PARA BIOLOGIA,OB,80,0,0,80,0,,,,1,48.01.002
3,GQI00050,QUÍMICA GERAL II,OB,60,0,0,60,0,,,,1,48.01.002
4,MGB00002,FUNDAMENTOS DE INICIACAO A PESQUISA,OB,40,0,0,40,10,,,,1,48.01.002


In [ ]:
df_bruto.pivot_table(index="curso", columns="tipo", values="codigo", aggfunc="count", fill_value=0)

tipo,AC,E,O,OB
curso,,,,
15.05.003,2,0,72,74
16.01.004,14,76,85,65
17.01.003,2,0,46,62
18.01.004,3,0,110,70
34.02.003,6,0,47,53
48.01.002,2,0,141,33


## 2. Transform

In [35]:
# Só com as obrigatórias

df = df_bruto[df_bruto["tipo"] == "OB"].copy()

for coluna in ["cht", "chp", "ches", "chtotal", "chex"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype(int)

df["chprereq"] = pd.to_numeric(df["chprereq"], errors="coerce")
df["periodo"] = df["periodo"].astype(int)

print("Obrigatórias:", df.shape)

Obrigatórias: (357, 13)


In [36]:
df["desativada"] = df["nome"].str.contains("(Desativada)", regex=False)
df["nome"] = df["nome"].str.replace("(Desativada)", "", regex=False).str.strip()

# Nome sem num ou sufixo, pra achar mesma materia em cursos diferentes
# ex.: ANATOMIA ANIMAL II -> ANATOMIA ANIMAL, BIOQUÍMICA-A -> BIOQUÍMICA
df["nomeBase"] = df["nome"].str.replace(r"(\s+[IVX]+)?(\s*-\s*[A-Z])?$", "", regex=True)

df[["codigo", "nome", "nomeBase", "desativada"]].sample(5, random_state=1)

,codigo,nome,nomeBase,desativada
718,MSV00014,DOENCAS INFECCIOSAS DOS ANIM. DOMESTICOS,DOENCAS INFECCIOSAS DOS ANIM. DOMESTICOS,False
679,MMO00084,ANATOMIA ANIMAL II,ANATOMIA ANIMAL,False
891,MOC00086,PERIODONTIA II - A,PERIODONTIA,False
435,MGM00411,PRIMEIROS SOCORROS,PRIMEIROS SOCORROS,False
879,MOC00082,RADIOLOGIA ODONTOLÓGICA E IMAGINOLOGIA II,RADIOLOGIA ODONTOLÓGICA E IMAGINOLOGIA,False


In [37]:
# separa pré e co-requisitos, pega só os cod de "[n - CODIGO] NOME"
df["preCodigos"] = df["prereq"].str.findall(r"([A-Z]{3}\d{5})\]")
df["coCodigos"] = df["coreq"].str.findall(r"([A-Z]{3}\d{5})\]")

# requisito de ex: "7º periodo completo"
df["periodosCompletos"] = df["prereq"].str.findall(r"(\d+)º período completo").apply(lambda numeros: [int(numero) for numero in numeros])

df[df["preCodigos"].str.len() > 1][["codigo", "prereq", "preCodigos"]].head()

,codigo,prereq,preCodigos
12,MFL00050,[2 - MGB00001] NEUROCIENCIAS [2 - MMO00038] MO...,"[MGB00001, MMO00038]"
24,MPT00063,[2 - GCM00017] BIOQUÍMICA I [3 - MPT00058] PAT...,"[GCM00017, MPT00058]"
25,MPT00064,[1 - GCM00016] BIOLOGIA CELULAR E MOLECULAR [2...,"[GCM00016, GCM00017, MFL00050, MPT00058]"
26,MPT00065,[3 - GIM00021] IMUNOLOGIA V [4 - MIP00045] VIR...,"[GIM00021, MIP00045]"
27,MPT00066,[4 - MIP00042] BACTERIOLOGIA V [4 - MIP00043] ...,"[MIP00042, MIP00043]"


In [ ]:

df["departamento"] = df["codigo"].str[:3]

In [ ]:

nos_curso = pd.DataFrame(cursos)
periodos = df.groupby("curso")["periodo"].max().rename("periodos")
nos_curso = nos_curso.merge(periodos, left_on="codigo", right_index=True)
nos_curso

,codigo,nome,chObrigatoria,periodos
0,48.01.002,BIOMEDICINA,3184,8
1,34.02.003,ENFERMAGEM,4145,10
2,15.05.003,FARMÁCIA,4317,10
3,16.01.004,MEDICINA,7001,12
4,18.01.004,MEDICINA VETERINÁRIA,5170,10
5,17.01.003,ODONTOLOGIA,4035,9


In [ ]:
nos_disciplina = (df.rename(columns={"chtotal": "chTotal"})
                  [["codigo", "nome", "nomeBase", "cht", "chp", "ches", "chTotal", "chex", "desativada"]]
                  .drop_duplicates("codigo"))

# confere se mesma disciplina n veio com dados dif em cursos dif
conflito = df.groupby("codigo")[["nome", "chtotal"]].nunique()
print("disciplinas:", len(nos_disciplina), "| conflitos:", int((conflito > 1).any(axis=1).sum()))

disciplinas: 356 | conflitos: 0


In [ ]:
nos_depto = pd.DataFrame({"sigla": sorted(df["departamento"].unique())})
len(nos_depto)

42

In [42]:
# relação: OFERECE (curso -> disciplina)

rel_oferece = df[["curso", "codigo", "periodo", "tipo", "chprereq", "periodosCompletos"]].rename(
    columns={"chprereq": "chMinima"})
len(rel_oferece)

357

In [43]:
# relação: REQUER (disciplina -> pré ou co-requisito)

pre = df[["codigo", "preCodigos"]].explode("preCodigos").dropna().rename(columns={"preCodigos": "requisito"})
pre["tipo"] = "pre"
co = df[["codigo", "coCodigos"]].explode("coCodigos").dropna().rename(columns={"coCodigos": "requisito"})
co["tipo"] = "co"
rel_requer = pd.concat([pre, co]).drop_duplicates()

# requisito que não existe como disciplina
orfaos = rel_requer[~rel_requer["requisito"].isin(nos_disciplina["codigo"])]
print("requer:", len(rel_requer), "| órfãos:", len(orfaos))

requer: 614 | órfãos: 0


In [44]:
# relação: DO_DEPTO

rel_depto = df[["codigo", "departamento"]].drop_duplicates()

In [45]:
# relação: EQUIVALE_A (opcional, lê data/equivalencias.csv se existir)

arq_eq = DATA_DIR / "equivalencias.csv"

if arq_eq.exists():
    rel_equivale = pd.read_csv(arq_eq, dtype=str)
    rel_equivale = rel_equivale.apply(lambda c: c.str.strip().str.upper())
    rel_equivale = rel_equivale[rel_equivale["codigo_a"] != rel_equivale["codigo_b"]].drop_duplicates()
    
    print("equivalências:", len(rel_equivale))

else:
    rel_equivale = pd.DataFrame(columns=["codigo_a", "codigo_b"])
    
    print("sem equivalencias.csv, pulando")

equivalências: 0


In [ ]:
# sugestão de equivalências pro grupo revisar: mesmo nomeBase em cursos diferentes
# (não é carregado, só salvo em data/processed)

pares = df[["curso", "codigo", "nome", "nomeBase"]].merge(df[["curso", "codigo", "nome", "nomeBase"]], on="nomeBase")
pares = pares[(pares["curso_x"] != pares["curso_y"]) & (pares["codigo_x"] < pares["codigo_y"])]
sugestao = pares.rename(columns={"codigo_x": "codigo_a", "codigo_y": "codigo_b", "nome_x": "nome_a", "nome_y": "nome_b"})
sugestao = sugestao[["codigo_a", "nome_a", "codigo_b", "nome_b"]].drop_duplicates()
sugestao.to_csv(OUT_DIR / "equivalencias_sugeridas.csv", index=False)
print("pares sugeridos:", len(sugestao))

pares sugeridos: 117


In [47]:
# salva as tabelas tratadas
for nome, tab in {"cursos": nos_curso, "disciplinas": nos_disciplina, "departamentos": nos_depto,
                  "oferece": rel_oferece, "requer": rel_requer, "do_depto": rel_depto}.items():
    tab.to_csv(OUT_DIR / f"{nome}.csv", index=False)

In [ ]:
# NaN -> None antes de mandar pro neo4j
def registros(tab):
    return tab.astype(object).where(tab.notna(), None).to_dict("records")

## 3. Load
1. **Schema:** executar `cypher/schema.cypher` (constraints e índices).
2. **Dados:** `MERGE` é idempotente (slide 30); valores sempre por parâmetro, nunca por concatenação (slide 42); `UNWIND` em lotes.

In [22]:
def executar_arquivo_cypher(caminho):
    texto = re.sub(r"//.*", "", Path(caminho).read_text(encoding="utf-8"))
    comandos = [c.strip() for c in texto.split(";") if c.strip()]
    for c in comandos:
        driver.execute_query(c, database_=NEO4J_DB)
    return len(comandos)

executar_arquivo_cypher(CYPHER_DIR / "schema.cypher")

5

In [23]:
# carga em lotes
def carregar(query, tab, lote=500):
    rows = registros(tab)
    for i in range(0, len(rows), lote):
        driver.execute_query(query, rows=rows[i:i + lote], database_=NEO4J_DB)
    return len(rows)

In [24]:
# nós
carregar("""
UNWIND $rows AS r
MERGE (c:Curso {codigo: r.codigo})
SET c.nome = r.nome, c.chObrigatoria = r.chObrigatoria, c.periodos = r.periodos
""", nos_curso)

carregar("""
UNWIND $rows AS r
MERGE (d:Disciplina {codigo: r.codigo})
SET d.nome = r.nome, d.nomeBase = r.nomeBase, d.cht = r.cht, d.chp = r.chp,
    d.ches = r.ches, d.chTotal = r.chTotal, d.chex = r.chex, d.desativada = r.desativada
""", nos_disciplina)

carregar("""
UNWIND $rows AS r
MERGE (:Departamento {sigla: r.sigla})
""", nos_depto)

42

In [25]:
# relações
carregar("""
UNWIND $rows AS r
MATCH (c:Curso {codigo: r.curso})
MATCH (d:Disciplina {codigo: r.codigo})
MERGE (c)-[o:OFERECE]->(d)
SET o.periodo = r.periodo, o.tipo = r.tipo, o.chMinima = r.chMinima, o.periodosCompletos = r.periodosCompletos
""", rel_oferece)

carregar("""
UNWIND $rows AS r
MATCH (d:Disciplina {codigo: r.codigo})
MATCH (req:Disciplina {codigo: r.requisito})
MERGE (d)-[:REQUER {tipo: r.tipo}]->(req)
""", rel_requer)

carregar("""
UNWIND $rows AS r
MATCH (d:Disciplina {codigo: r.codigo})
MATCH (dp:Departamento {sigla: r.departamento})
MERGE (d)-[:DO_DEPTO]->(dp)
""", rel_depto)

carregar("""
UNWIND $rows AS r
MATCH (a:Disciplina {codigo: r.codigo_a})
MATCH (b:Disciplina {codigo: r.codigo_b})
MERGE (a)-[:EQUIVALE_A]->(b)
""", rel_equivale)

0

## 4. Validação

In [26]:
def consulta(q, **params):
    recs, _, _ = driver.execute_query(q, params, database_=NEO4J_DB)
    return pd.DataFrame([r.data() for r in recs])

In [27]:
# 1. nós por label
consulta("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS total ORDER BY label")

,label,total
0,Curso,6
1,Departamento,42
2,Disciplina,356


In [28]:
# 2. relações por tipo
consulta("MATCH ()-[r]->() RETURN type(r) AS relacao, count(*) AS total ORDER BY relacao")

,relacao,total
0,DO_DEPTO,356
1,OFERECE,357
2,REQUER,614


In [29]:
# 3. disciplinas por curso: pdf x grafo
grafo = consulta("MATCH (c:Curso)-[:OFERECE]->(d) RETURN c.codigo AS curso, count(d) AS grafo")
pdf_qtd = df.groupby("curso").size().rename("pdf").reset_index()
v3 = pdf_qtd.merge(grafo, on="curso")
v3["ok"] = v3["pdf"] == v3["grafo"]
v3

,curso,pdf,grafo,ok
0,15.05.003,74,74,True
1,16.01.004,65,65,True
2,17.01.003,62,62,True
3,18.01.004,70,70,True
4,34.02.003,53,53,True
5,48.01.002,33,33,True


In [30]:
# 4. carga horária somada x carga obrigatória impressa no pdf
consulta("""
MATCH (c:Curso)-[:OFERECE]->(d:Disciplina)
WITH c, sum(d.chTotal) AS soma
RETURN c.nome AS curso, c.chObrigatoria AS pdf, soma AS grafo, soma = c.chObrigatoria AS ok
ORDER BY curso
""")

,curso,pdf,grafo,ok
0,BIOMEDICINA,3184,3184,True
1,ENFERMAGEM,4145,4145,True
2,FARMÁCIA,4317,4317,True
3,MEDICINA,7001,7001,True
4,MEDICINA VETERINÁRIA,5170,5170,True
5,ODONTOLOGIA,4035,4035,True


In [31]:
# 5. nenhum código duplicado
consulta("""
MATCH (d:Disciplina)
WITH d.codigo AS codigo, count(*) AS n
WHERE n > 1
RETURN count(*) AS duplicados
""")

,duplicados
0,0


In [32]:
# 6. nenhum requisito perdido: requer no grafo = requer no pandas
r = consulta("MATCH ()-[r:REQUER]->() RETURN count(r) AS grafo")
print("pandas:", len(rel_requer), "| grafo:", int(r["grafo"][0]), "| órfãos:", len(orfaos))

pandas: 614 | grafo: 614 | órfãos: 0


In [33]:
# 7. nenhum campo vazio onde não pode
consulta("""
MATCH (d:Disciplina)
WHERE d.nome IS NULL OR d.nome = '' OR d.chTotal IS NULL OR NOT (d)-[:DO_DEPTO]->()
RETURN count(d) AS problemas
""")

,problemas
0,0


In [34]:
driver.close()